In [2]:
# ── Imports & Configuration ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, glob, warnings
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW  = os.path.join(BASE, "data", "raw")
SEASON_TAG  = "serie_a_2025_2026"
SEASON_LABEL = "2025_2026"

CM_POSITIONS = {"AMC", "AMR", "AML"}
MIN_MINUTES  = 900

PILLAR_WEIGHTS = {
    "Line-Breaking Passing": 0.35,
    "Chance Creation":       0.30,
    "Reception & Retention": 0.20,
    "Carrying & Progression":0.15,
}

KPI_MAP = {
    "Line-Breaking Passing": {
        "through_balls_p90":      {"weight": 0.40, "negative": False},
        "final_third_passes_p90": {"weight": 0.35, "negative": False},
        "passes_into_box_p90":    {"weight": 0.25, "negative": False},
    },
    "Chance Creation": {
        "key_passes_p90":          {"weight": 0.40, "negative": False},
        "big_chances_created_p90": {"weight": 0.35, "negative": False},
        "crosses_into_box_p90":    {"weight": 0.25, "negative": False},
    },
    "Reception & Retention": {
        "pass_completion_pct":   {"weight": 0.35, "negative": False},
        "successful_dribbles_p90":{"weight": 0.35, "negative": False},
        "turnovers_p90":         {"weight": 0.30, "negative": True},
    },
    "Carrying & Progression": {
        "take_ons_won_p90":       {"weight": 0.35, "negative": False},
        "progressive_passes_p90":{"weight": 0.40, "negative": False},
        "switches_p90":          {"weight": 0.25, "negative": False},
    },
}

KPI_LABELS = {
    "through_balls_p90":       "Through Balls",
    "final_third_passes_p90":  "Final Third Passes",
    "passes_into_box_p90":     "Passes into Box",
    "key_passes_p90":          "Key Passes (Assists)",
    "big_chances_created_p90": "Big Chances Created",
    "crosses_into_box_p90":    "Crosses into Box",
    "pass_completion_pct":     "Pass Completion %",
    "successful_dribbles_p90": "Successful Dribbles",
    "turnovers_p90":           "Turnovers (neg.)",
    "take_ons_won_p90":        "Take-Ons Won",
    "progressive_passes_p90":  "Progressive Passes",
    "switches_p90":            "Switches of Play",
}

print(f"Target season: {SEASON_TAG}")
print(f"Events dir exists: {os.path.isdir(os.path.join(RAW, SEASON_TAG, 'events'))}")
n_files = len(glob.glob(os.path.join(RAW, SEASON_TAG, "events", "*.csv")))
print(f"Match files: {n_files}")
print("Setup complete ✓")

Target season: serie_a_2025_2026
Events dir exists: True
Match files: 319
Setup complete ✓


In [3]:
# ── Load 2025/26 events ──────────────────────────────────────────────────────
evt_dir = os.path.join(RAW, SEASON_TAG, "events")
files = sorted(glob.glob(os.path.join(evt_dir, "*.csv")))
print(f"Loading {len(files)} match files...")

frames = []
for f in files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df["season"] = SEASON_LABEL
        df["gameweek"] = os.path.basename(f).split("_")[0]
        frames.append(df)
    except:
        continue

all_events = pd.concat(frames, ignore_index=True)
cm_events  = all_events[all_events["position"].isin(CM_POSITIONS)].copy()

print(f"Total events: {len(all_events):,}")
print(f"Creative MF events: {len(cm_events):,}")
print(f"Unique creative MF players: {cm_events['player_name'].nunique()}")

Loading 319 match files...
Total events: 532,428
Creative MF events: 0
Unique creative MF players: 0
Total events: 532,428
Creative MF events: 0
Unique creative MF players: 0


In [4]:
# ── Estimate minutes & compute raw KPIs ──────────────────────────────────────

# --- Minutes ---
dm = cm_events.copy()
dm["time_min"] = pd.to_numeric(dm["time_min"], errors="coerce")

player_match = (
    dm.groupby(["player_name", "player_id", "team_name", "season", "match_id"])
    ["time_min"].max().reset_index().rename(columns={"time_min": "last_event_min"})
)
minutes_df = (
    player_match.groupby(["player_name", "player_id", "team_name", "season"])
    .agg(matches=("match_id", "nunique"), total_minutes=("last_event_min", "sum"))
    .reset_index()
)

# --- Raw KPIs ---
df = cm_events.copy()
for c in ["type_id", "outcome", "x", "y"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["Pass End X"] = pd.to_numeric(df["Pass End X"], errors="coerce")
df["Pass End Y"] = pd.to_numeric(df["Pass End Y"], errors="coerce")

grp = ["player_name", "player_id", "team_name", "season"]

def _cnt(mask):
    return df[mask].groupby(grp).size()

kpis = pd.DataFrame()

# ── Line-Breaking Passing ──
# Through balls: pass (type 1) with qualifier "Through ball" != N/A
kpis["through_balls"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Through ball"] != "N/A") & df["Through ball"].notna()
)
# Final third passes: successful passes ending in final third (Pass End X > 66.7)
kpis["final_third_passes"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Pass End X"] > 66.7)
)
# Passes into box: successful passes ending in penalty area (Pass End X > 83, 21 < Pass End Y < 79)
kpis["passes_into_box"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) &
    (df["Pass End X"] > 83) & (df["Pass End Y"] > 21) & (df["Pass End Y"] < 79)
)

# ── Chance Creation ──
# Key passes (assists): qualifier 210 = Assist
kpis["key_passes"] = _cnt(
    (df["type_id"] == 1) & (df["Assist"] != "N/A") & df["Assist"].notna()
)
# Big chances created: passes that led to a shot with Big Chance qualifier
# We approximate by looking at passes with Intentional Assist + link to big chance
# Simpler: count passes with qualifier 210 where related shot had qualifier 214
# For simplicity, use Related event ID to link assists to big-chance shots
kpis["big_chances_created"] = _cnt(
    (df["type_id"] == 1) & (df["Assist"] != "N/A") & df["Assist"].notna() &
    (df["Intentional Assist"] != "N/A") & df["Intentional Assist"].notna()
)
# Crosses into box: cross qualifier (Q2) with outcome 1
kpis["crosses_into_box"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) & (df["Cross"] != "N/A") & df["Cross"].notna()
)

# ── Reception & Retention ──
kpis["passes_completed"]  = _cnt((df["type_id"] == 1) & (df["outcome"] == 1))
kpis["passes_total"]      = _cnt(df["type_id"] == 1)
kpis["successful_dribbles"]= _cnt((df["type_id"] == 3) & (df["outcome"] == 1))
kpis["turnovers"]         = _cnt((df["type_id"] == 50) | ((df["type_id"] == 61) & (df["outcome"] == 0)))

# ── Carrying & Progression ──
kpis["take_ons_won"]  = _cnt((df["type_id"] == 3) & (df["outcome"] == 1))
# Progressive passes: successful passes that move ball ≥25% closer to goal (x increases by ≥10)
kpis["progressive_passes"] = _cnt(
    (df["type_id"] == 1) & (df["outcome"] == 1) &
    ((df["Pass End X"] - df["x"]) >= 10)
)
# Switches of play: qualifier 196
kpis["switches"] = _cnt(
    (df["type_id"] == 1) & (df["Switch of play"] != "N/A") & df["Switch of play"].notna()
)

kpis = kpis.fillna(0).reset_index()

# --- Merge & per-90 ---
merged = kpis.merge(minutes_df[["player_name","player_id","season","matches","total_minutes"]],
                     on=["player_name","player_id","season"], how="left")
player_stats = merged[merged["total_minutes"] >= MIN_MINUTES].copy()

per90_cols = ["through_balls","final_third_passes","passes_into_box",
              "key_passes","big_chances_created","crosses_into_box",
              "successful_dribbles","turnovers","take_ons_won","progressive_passes","switches"]
for c in per90_cols:
    player_stats[f"{c}_p90"] = (player_stats[c] / player_stats["total_minutes"]) * 90

player_stats["pass_completion_pct"] = (player_stats["passes_completed"]/player_stats["passes_total"]*100).fillna(0)

print(f"Players with ≥{MIN_MINUTES} min: {len(player_stats)}")
player_stats.sort_values("key_passes_p90", ascending=False)[
    ["player_name","team_name","matches","total_minutes",
     "key_passes_p90","through_balls_p90","final_third_passes_p90","progressive_passes_p90"]
].head(10).round(2)

KeyError: 'player_name'

In [ ]:
# ── Possession-Based Normalization ──────────────────────────────────────────────
ae = all_events.copy()
ae["type_id"] = pd.to_numeric(ae["type_id"], errors="coerce")
ae["outcome"] = pd.to_numeric(ae["outcome"], errors="coerce")

team_passes = (
    ae[(ae["type_id"] == 1) & (ae["outcome"] == 1)]
    .groupby(["match_id", "team_name"]).size()
    .reset_index(name="passes")
)

match_teams = team_passes.groupby("match_id")["team_name"].apply(list).reset_index()
match_teams = match_teams[match_teams["team_name"].apply(len) == 2]

poss_rows = []
for _, r in match_teams.iterrows():
    mid = r["match_id"]
    t1, t2 = r["team_name"]
    mp = team_passes[team_passes["match_id"] == mid].set_index("team_name")["passes"]
    total = mp.get(t1, 0) + mp.get(t2, 0)
    if total > 0:
        poss_rows.append({"team_name": t1, "match_id": mid, "poss_pct": mp.get(t1, 0) / total * 100})
        poss_rows.append({"team_name": t2, "match_id": mid, "poss_pct": mp.get(t2, 0) / total * 100})

poss_df = pd.DataFrame(poss_rows)
team_poss = poss_df.groupby("team_name")["poss_pct"].mean().reset_index(name="avg_possession")
league_avg_poss = team_poss["avg_possession"].mean()
team_poss["adj_factor"] = team_poss["avg_possession"] / league_avg_poss

print(f"League-avg possession: {league_avg_poss:.1f}%")

# ── Apply adjustment to count-based p90 metrics ──────────────────────────────
player_stats = player_stats.merge(team_poss[["team_name", "avg_possession", "adj_factor"]], on="team_name", how="left")

count_based_p90 = [
    "through_balls_p90", "final_third_passes_p90", "passes_into_box_p90",
    "key_passes_p90", "big_chances_created_p90", "crosses_into_box_p90",
    "successful_dribbles_p90", "turnovers_p90",
    "take_ons_won_p90", "progressive_passes_p90", "switches_p90",
]

for c in count_based_p90:
    player_stats[f"{c}_raw"] = player_stats[c]
    player_stats[c] = player_stats[c] * player_stats["adj_factor"]

print(f"\nAdjustment applied to {len(count_based_p90)} count-based p90 metrics.")
print(f"Pass Completion % NOT adjusted (already a ratio).")
player_stats.sort_values("key_passes_p90", ascending=False)[
    ["player_name", "team_name", "avg_possession", "adj_factor", "key_passes_p90"]
].head(10).round(2)

In [ ]:
# ── Percentiles, Pillar Scores, Composite Score ──────────────────────────────

ps = player_stats.copy()

# ── Percentile ranks ─────────────────────────────────────────
for pillar, kpis_def in KPI_MAP.items():
    for kpi, info in kpis_def.items():
        pc = f"{kpi}_pct"
        ps[pc] = ps[kpi].rank(pct=True) * 100
        if info["negative"]:
            ps[pc] = 100 - ps[pc]

# ── Pillar scores ─────────────────────────────────────────
for pillar, kpis_def in KPI_MAP.items():
    col = f"pillar_{pillar.replace(' ','_').replace('&','and').lower()}"
    ps[col] = sum(ps[f"{k}_pct"] * v["weight"] for k, v in kpis_def.items())

# ── Composite score ──────────────────────────────────────────
ps["composite_score"] = (
    ps["pillar_line-breaking_passing"]  * 0.35 +
    ps["pillar_chance_creation"]        * 0.30 +
    ps["pillar_reception_and_retention"]* 0.20 +
    ps["pillar_carrying_and_progression"]* 0.15
)
ps = ps.sort_values("composite_score", ascending=False).reset_index(drop=True)
ps["rank"] = range(1, len(ps)+1)

print(f"Final scored players: {len(ps)}")
ps[["rank","player_name","team_name","matches","total_minutes","composite_score",
    "pillar_line-breaking_passing","pillar_chance_creation",
    "pillar_reception_and_retention","pillar_carrying_and_progression"]].head(15).round(1)

## 📊 Interactive Composite Ranking

Bar chart of the top 25 players ranked by composite score, with pillar breakdown on hover.

In [ ]:
# ── Interactive stacked bar: Top 25 by composite score ────────────────────────

top_n = ps.head(25).copy().sort_values("composite_score", ascending=True)

pillar_info = [
    ("pillar_line-breaking_passing",     "Line-Breaking Passing (0.35)",  "#1f77b4"),
    ("pillar_chance_creation",           "Chance Creation (0.30)",        "#ff7f0e"),
    ("pillar_reception_and_retention",   "Reception & Retention (0.20)",  "#2ca02c"),
    ("pillar_carrying_and_progression",  "Carrying & Progression (0.15)", "#d62728"),
]

fig_bar = go.Figure()
for col, name, color in pillar_info:
    weight = {"pillar_line-breaking_passing":0.35, "pillar_chance_creation":0.30,
              "pillar_reception_and_retention":0.20, "pillar_carrying_and_progression":0.15}[col]
    fig_bar.add_trace(go.Bar(
        y=top_n["player_name"] + " (" + top_n["team_name"] + ")",
        x=(top_n[col] * weight).round(1),
        name=name, orientation="h", marker_color=color,
        hovertemplate="%{y}<br>" + name + ": %{x:.1f}<extra></extra>"
    ))

fig_bar.update_layout(
    barmode="stack",
    title="Top 25 Creative Midfielders — Serie A 2025/26<br><sub>Weighted Composite Score (pillar breakdown)</sub>",
    xaxis_title="Composite Score", yaxis_title="",
    height=700, template="plotly_white",
    legend=dict(orientation="h", y=-0.12, x=0.5, xanchor="center"),
    margin=dict(l=200),
)
fig_bar.show()

## 📈 Quantity (/90) vs Quality (Percentile) — Dual View

For each KPI we plot the **per-90 value** (x-axis = how often) against the **percentile rank** (y-axis = how good relative to peers). Top-right quadrant = high quantity AND high quality.

In [ ]:
# ── Quantity vs Quality scatter plots ─────────────────────────────────────────

qty_qual_kpis = [
    ("through_balls_p90",       "through_balls_p90_pct",       "Through Balls"),
    ("final_third_passes_p90",  "final_third_passes_p90_pct",  "Final Third Passes"),
    ("passes_into_box_p90",     "passes_into_box_p90_pct",     "Passes into Box"),
    ("key_passes_p90",          "key_passes_p90_pct",          "Key Passes"),
    ("big_chances_created_p90", "big_chances_created_p90_pct", "Big Chances Created"),
    ("crosses_into_box_p90",    "crosses_into_box_p90_pct",    "Crosses into Box"),
    ("successful_dribbles_p90", "successful_dribbles_p90_pct", "Successful Dribbles"),
    ("progressive_passes_p90",  "progressive_passes_p90_pct",  "Progressive Passes"),
    ("turnovers_p90",           "turnovers_p90_pct",           "Turnovers (inv.)"),
]

n = len(qty_qual_kpis)
rows, cols = 3, 3
fig_qq = make_subplots(rows=rows, cols=cols,
                       subplot_titles=[t[2] for t in qty_qual_kpis],
                       horizontal_spacing=0.08, vertical_spacing=0.10)

top10_names = set(ps.head(10)["player_name"])

for i, (p90_col, pct_col, label) in enumerate(qty_qual_kpis):
    r, c = i // cols + 1, i % cols + 1
    
    others = ps[~ps["player_name"].isin(top10_names)]
    fig_qq.add_trace(go.Scatter(
        x=others[p90_col], y=others[pct_col],
        mode="markers", marker=dict(size=6, color="lightgrey", opacity=0.6),
        text=others["player_name"] + " (" + others["team_name"] + ")",
        hovertemplate="%{text}<br>" + label + " p90: %{x:.2f}<br>Percentile: %{y:.0f}<extra></extra>",
        showlegend=False,
    ), row=r, col=c)
    
    top = ps[ps["player_name"].isin(top10_names)]
    fig_qq.add_trace(go.Scatter(
        x=top[p90_col], y=top[pct_col],
        mode="markers+text", marker=dict(size=10, color="#8a1f33"),
        text=top["player_name"].str.split().str[-1],
        textposition="top center", textfont=dict(size=8),
        hovertemplate="%{customdata}<br>" + label + " p90: %{x:.2f}<br>Percentile: %{y:.0f}<extra></extra>",
        customdata=top["player_name"] + " (" + top["team_name"] + ")",
        showlegend=False,
    ), row=r, col=c)
    
    fig_qq.add_hline(y=50, line_dash="dot", line_color="grey", opacity=0.4, row=r, col=c)
    fig_qq.add_vline(x=ps[p90_col].median(), line_dash="dot", line_color="grey", opacity=0.4, row=r, col=c)

fig_qq.update_layout(
    title="Quantity (p90) vs Quality (Percentile) — All KPIs<br><sub>Top 10 composite players highlighted in red | Possession-adjusted</sub>",
    height=900, template="plotly_white", showlegend=False,
)
fig_qq.update_xaxes(title_text="Per 90", title_font_size=9)
fig_qq.update_yaxes(title_text="Percentile", title_font_size=9)
fig_qq.show()

## 📋 Player Scouting Card — Quantity & Quality

Select a player from the dropdown to see their full KPI breakdown: raw per-90 value (quantity), percentile rank (quality), and pillar scores. Color-coded by percentile tier.

In [ ]:
# ── Player scouting card — dropdown-based ──────────────────────────────────────

qty_qual_pairs = [
    ("through_balls_p90",       "through_balls_p90_pct",       "Through Balls",         "Line-Breaking Passing"),
    ("final_third_passes_p90",  "final_third_passes_p90_pct",  "Final Third Passes",    "Line-Breaking Passing"),
    ("passes_into_box_p90",     "passes_into_box_p90_pct",     "Passes into Box",       "Line-Breaking Passing"),
    ("key_passes_p90",          "key_passes_p90_pct",          "Key Passes (Assists)",  "Chance Creation"),
    ("big_chances_created_p90", "big_chances_created_p90_pct", "Big Chances Created",   "Chance Creation"),
    ("crosses_into_box_p90",    "crosses_into_box_p90_pct",    "Crosses into Box",      "Chance Creation"),
    ("pass_completion_pct",     "pass_completion_pct_pct",     "Pass Completion %",     "Reception & Retention"),
    ("successful_dribbles_p90", "successful_dribbles_p90_pct", "Successful Dribbles",   "Reception & Retention"),
    ("turnovers_p90",           "turnovers_p90_pct",           "Turnovers (inv.)",      "Reception & Retention"),
    ("take_ons_won_p90",        "take_ons_won_p90_pct",        "Take-Ons Won",          "Carrying & Progression"),
    ("progressive_passes_p90",  "progressive_passes_p90_pct",  "Progressive Passes",    "Carrying & Progression"),
    ("switches_p90",            "switches_p90_pct",            "Switches of Play",      "Carrying & Progression"),
]

pillar_colors = {
    "Line-Breaking Passing":   "#1f77b4",
    "Chance Creation":         "#ff7f0e",
    "Reception & Retention":   "#2ca02c",
    "Carrying & Progression":  "#d62728",
}

def pctl_bg(v):
    if pd.isna(v): return "white"
    if v >= 80: return "rgba(0,180,80,0.25)"
    if v >= 60: return "rgba(0,180,80,0.10)"
    if v >= 40: return "white"
    if v >= 20: return "rgba(255,80,80,0.10)"
    return "rgba(255,80,80,0.25)"

def build_player_table(row):
    kpi_names, raw_vals, pct_vals, pillar_names = [], [], [], []
    pct_colors = []
    for raw_c, pct_c, label, pillar in qty_qual_pairs:
        kpi_names.append(label)
        rv = row.get(raw_c, np.nan)
        pv = row.get(pct_c, np.nan)
        raw_vals.append(f"{rv:.2f}" if pd.notna(rv) else "—")
        pct_vals.append(f"{pv:.0f}" if pd.notna(pv) else "—")
        pct_colors.append(pctl_bg(pv))
        pillar_names.append(pillar)

    pillar_cell_colors = [pillar_colors.get(p, "white") for p in pillar_names]
    pillar_fill = [f"rgba({int(c[1:3],16)},{int(c[3:5],16)},{int(c[5:7],16)},0.12)"
                   if c.startswith("#") else "white" for c in pillar_cell_colors]

    return go.Table(
        header=dict(
            values=["Pillar", "KPI", "Value (/90)", "Percentile"],
            fill_color="#8a1f33", font=dict(color="white", size=11),
            align="center", height=30,
        ),
        cells=dict(
            values=[pillar_names, kpi_names, raw_vals, pct_vals],
            fill_color=[pillar_fill, ["white"]*len(kpi_names), ["white"]*len(kpi_names), pct_colors],
            align=["left", "left", "center", "center"],
            font=dict(size=11), height=26,
        ),
    )

fig_card = go.Figure()
player_list_t = ps.sort_values("rank")

for i, (_, row) in enumerate(player_list_t.iterrows()):
    tbl = build_player_table(row)
    tbl.visible = True if i == 0 else False
    fig_card.add_trace(tbl)

buttons_t = []
for i, (_, row) in enumerate(player_list_t.iterrows()):
    vis = [j == i for j in range(len(player_list_t))]
    pillar_summary = (
        f"Line-Break: {row['pillar_line-breaking_passing']:.0f} | "
        f"Create: {row['pillar_chance_creation']:.0f} | "
        f"Retain: {row['pillar_reception_and_retention']:.0f} | "
        f"Carry: {row['pillar_carrying_and_progression']:.0f}"
    )
    buttons_t.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis},
              {"title": f"Scouting Card — #{row['rank']} {row['player_name']} ({row['team_name']})<br>"
                        f"<sub>Composite: {row['composite_score']:.1f} | "
                        f"{row['matches']:.0f} GP, {row['total_minutes']:.0f} min | "
                        f"Poss: {row.get('avg_possession',50):.1f}% | {pillar_summary}</sub>"}],
    ))

first = player_list_t.iloc[0]
fig_card.update_layout(
    title=f"Scouting Card — #1 {first['player_name']} ({first['team_name']})<br>"
          f"<sub>Composite: {first['composite_score']:.1f} | "
          f"{first['matches']:.0f} GP, {first['total_minutes']:.0f} min</sub>",
    height=500, margin=dict(l=10, r=10, t=100, b=10),
    updatemenus=[dict(
        buttons=buttons_t, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.18, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_card.show()

## 🎯 Interactive Player Explorer

Select any player from the dropdown to see their full detailed radar alongside the league-average benchmark.

In [ ]:
# ── Player explorer with dropdown ─────────────────────────────────────────────

radar_kpis = [
    ("through_balls_p90_pct",       "Through Balls"),
    ("final_third_passes_p90_pct",  "Final Third Passes"),
    ("passes_into_box_p90_pct",     "Passes into Box"),
    ("key_passes_p90_pct",          "Key Passes"),
    ("big_chances_created_p90_pct", "Big Chances Created"),
    ("crosses_into_box_p90_pct",    "Crosses into Box"),
    ("pass_completion_pct_pct",     "Pass Compl. %"),
    ("successful_dribbles_p90_pct", "Successful Dribbles"),
    ("turnovers_p90_pct",           "Turnovers (inv.)"),
    ("take_ons_won_p90_pct",        "Take-Ons Won"),
    ("progressive_passes_p90_pct",  "Progressive Passes"),
    ("switches_p90_pct",            "Switches of Play"),
]
radar_cols   = [c for c, _ in radar_kpis]
radar_labels = [l for _, l in radar_kpis]

fig_explore = go.Figure()

avg_vals = [50] * len(radar_labels) + [50]
fig_explore.add_trace(go.Scatterpolar(
    r=avg_vals, theta=radar_labels + [radar_labels[0]],
    line=dict(color="grey", width=1, dash="dot"), fill="none",
    name="League Average (50th pctl)", visible=True,
))

player_list = ps.sort_values("rank")

for i, (_, row) in enumerate(player_list.iterrows()):
    vals = [row.get(c, 50) for c in radar_cols]
    vals = [v if pd.notna(v) else 50 for v in vals]
    vals_closed = vals + [vals[0]]
    
    raw_metrics = [c.replace("_pct","") for c, _ in radar_kpis]
    hover = []
    for (pct_c, lbl), raw_c in zip(radar_kpis, raw_metrics):
        raw_v = row.get(raw_c, np.nan)
        pct_v = row.get(pct_c, np.nan)
        raw_str = f"{raw_v:.2f}" if pd.notna(raw_v) else "N/A"
        pct_str = f"{pct_v:.0f}" if pd.notna(pct_v) else "N/A"
        hover.append(f"{lbl}: {raw_str} p90 | {pct_str}th pctl")
    hover.append(hover[0])
    
    fig_explore.add_trace(go.Scatterpolar(
        r=vals_closed, theta=radar_labels + [radar_labels[0]],
        fill="toself", fillcolor="rgba(138,31,51,0.12)",
        line=dict(color="#8a1f33", width=2.5),
        text=hover, hovertemplate="%{text}<extra></extra>",
        name=f"#{row['rank']} {row['player_name']}",
        visible=True if i == 0 else False,
    ))

buttons = []
for i, (_, row) in enumerate(player_list.iterrows()):
    vis = [True] + [j == i for j in range(len(player_list))]
    buttons.append(dict(
        label=f"#{row['rank']} {row['player_name']} ({row['team_name']})",
        method="update",
        args=[{"visible": vis},
              {"title": f"Player Explorer — #{row['rank']} {row['player_name']} ({row['team_name']})"
                        f"<br><sub>Composite: {row['composite_score']:.1f} | "
                        f"{row['matches']:.0f} GP, {row['total_minutes']:.0f} min | "
                        f"Poss: {row.get('avg_possession',50):.1f}% (adj: {row.get('adj_factor',1):.2f})</sub>"}],
    ))

fig_explore.update_layout(
    title="Player Explorer — Select a Creative MF to view detailed radar<br>"
          "<sub>Hover for per-90 (quantity) + percentile (quality) for each KPI</sub>",
    polar=dict(radialaxis=dict(range=[0, 100], showticklabels=True, tickfont_size=8),
               angularaxis=dict(tickfont_size=9)),
    height=650, template="plotly_white",
    updatemenus=[dict(
        buttons=buttons, direction="down", showactive=True,
        x=0.02, xanchor="left", y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#8a1f33",
    )],
)
fig_explore.show()